## 1. Import Libraries and Check GPU

In [7]:
import cv2
import torch
import numpy as np
from pathlib import Path
import time
from typing import List, Dict
import matplotlib.pyplot as plt
from IPython.display import Video, display
from ultralytics.models.sam import SAM3SemanticPredictor

# Check GPU availability
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

Using device: mps


## 2. Initialize SAM3 Predictor

In [ ]:
# Initialize SAM3 semantic predictor
overrides = dict(
    conf=0.25,              # Confidence threshold
    task="segment",
    mode="predict",
    model="/Users/tommy/Projects/models/sam3.pt",  # Full path to model
    half=True,              # Use FP16 for faster inference on GPU
    save=False,
    verbose=False
)

# Use SAM3SemanticPredictor for frame-by-frame (webcam) processing
predictor = SAM3SemanticPredictor(overrides=overrides)

print("SAM3 loaded successfully!")

SAM3 loaded successfully!


## 3. Define Clothing Prompts

In [3]:
# Define which clothing items to detect
selected_prompts = [
   "clothes on ground",
   "pile of clothes",
   "dropped clothing"
]

print(f"Configured {len(selected_prompts)} clothing prompts")

Configured 3 clothing prompts


## 4. Optimized Segmenter Class (with Mask Tracking)

This class segments once, then tracks masks across frames using optical flow.
Only re-segments when tracking fails or scene changes significantly.

In [ ]:
import colorsys


class OptimizedSAM3Segmenter:
    """
    Real-time optimized segmentation with mask tracking.
    - Segments with SAM3 on the first frame, on a periodic refresh, on a large
      scene change, and whenever mask tracking is judged to have failed
    - Tracks masks between segmentations using dense optical flow

    Masks and boxes are stored in FULL-FRAME pixel coordinates, so tracking and
    rendering never have to reason about the inference scale.
    """

    def __init__(
        self,
        predictor,
        clothing_prompts: List[str],
        device='cuda',
        capture_size=(640, 480),      # Lower resolution for capture
        inference_size=320,            # Even lower for SAM3 inference
        resegment_interval=150,        # Re-run SAM3 every N frames (5 sec @ 30fps)
        scene_change_threshold=0.3,    # Re-segment if >30% of pixels change
        pixel_change_threshold=25,     # Grey levels a pixel must move to count as changed
        mask_area_tolerance=2.0,       # Max area drift of a tracked mask before giving up
        use_optical_flow=True
    ):
        """
        Args:
            predictor: SAM3SemanticPredictor instance
            clothing_prompts: List of text prompts
            device: 'cuda', 'mps', or 'cpu'
            capture_size: (width, height) for webcam capture
            inference_size: Long-side size for SAM3 inference (smaller = faster)
            resegment_interval: Re-run SAM3 every N frames
            scene_change_threshold: Fraction of pixels that must change appreciably
                before forcing a re-segmentation
            pixel_change_threshold: Absolute intensity change that counts as "changed"
            mask_area_tolerance: A tracked mask may grow or shrink by at most this
                factor relative to the last segmentation before tracking is
                declared lost
            use_optical_flow: Use optical flow for mask tracking
        """
        self.predictor = predictor
        self.clothing_prompts = clothing_prompts
        self.device = device
        self.capture_size = capture_size
        self.inference_size = inference_size
        self.resegment_interval = resegment_interval
        self.scene_change_threshold = scene_change_threshold
        self.pixel_change_threshold = pixel_change_threshold
        self.mask_area_tolerance = mask_area_tolerance
        self.use_optical_flow = use_optical_flow

        # Tracking state (all at full frame resolution)
        self.masks = None          # (N, H, W) float32, binary valued
        self.boxes = None          # (N, 4) xyxy in full-frame pixels
        self.class_ids = None
        self.ref_areas = None      # Mask areas at the last segmentation
        self.prev_gray = None
        self.frame_count = 0
        self.last_segment_frame = -999

        # Performance metrics
        self.fps_history = []
        self.segment_times = []
        self.track_times = []
        self.segment_count = 0
        self.track_count = 0
        self.track_failures = 0

        # Color map
        self.color_map = {}
        for i, prompt in enumerate(clothing_prompts):
            hue = (i * 137.5) % 360
            r, g, b = colorsys.hsv_to_rgb(hue / 360, 0.8, 0.9)
            self.color_map[i] = [int(r * 255), int(g * 255), int(b * 255)]

    def _run_segmentation(self, frame_small: np.ndarray, frame_shape) -> bool:
        """Run SAM3 and store masks/boxes in full-frame coordinates"""
        h, w = frame_shape[:2]
        sh, sw = frame_small.shape[:2]

        self.predictor.set_image(frame_small)
        results = self.predictor(text=self.clothing_prompts)

        if not results:
            return False

        result = results[0]
        if not hasattr(result, 'masks') or result.masks is None:
            return False

        masks_small = result.masks.data.cpu().numpy()
        if len(masks_small) == 0:
            return False

        # Upscale masks to full frame size once, here, so tracking and rendering
        # all operate in the same coordinate space.
        self.masks = np.stack([
            (cv2.resize(m.astype(np.float32), (w, h)) > 0.5).astype(np.float32)
            for m in masks_small
        ])

        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.cpu().numpy().astype(np.float32)
            # Scale from the ACTUAL small-frame size. inference_size is only the
            # long side, so using it for both axes squashes boxes on the short one.
            boxes[:, [0, 2]] *= w / sw
            boxes[:, [1, 3]] *= h / sh
            self.boxes = boxes
            if hasattr(result.boxes, 'cls'):
                self.class_ids = result.boxes.cls.cpu().numpy().astype(int)
            else:
                self.class_ids = np.zeros(len(self.masks), dtype=int)
        else:
            self.boxes = None
            self.class_ids = np.zeros(len(self.masks), dtype=int)

        self.ref_areas = self.masks.reshape(len(self.masks), -1).sum(axis=1)
        self.segment_count += 1
        return True

    @staticmethod
    def _boxes_from_masks(masks: np.ndarray) -> np.ndarray:
        """Tight xyxy boxes around each binary mask"""
        boxes = []
        for mask in masks:
            ys, xs = np.nonzero(mask > 0.5)
            if len(xs) == 0:
                boxes.append([0, 0, 0, 0])
            else:
                boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
        return np.array(boxes, dtype=np.float32)

    def _track_masks(self, curr_gray: np.ndarray, prev_gray: np.ndarray) -> bool:
        """
        Warp masks from the previous frame into the current one.

        cv2.remap is a BACKWARD warp: for every destination pixel it needs the
        location to sample in the source. Flow computed curr -> prev is exactly
        that map, which is why the frames are passed in that order.
        """
        if self.masks is None or len(self.masks) == 0:
            return False

        h, w = curr_gray.shape

        try:
            flow = cv2.calcOpticalFlowFarneback(
                curr_gray, prev_gray, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
        except cv2.error as e:
            print(f"Optical flow failed: {e}")
            return False

        h_idx, w_idx = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
        map_x = np.clip(w_idx + flow[..., 0], 0, w - 1).astype(np.float32)
        map_y = np.clip(h_idx + flow[..., 1], 0, h - 1).astype(np.float32)

        # Re-binarize after warping: repeated bilinear sampling would otherwise
        # blur the masks away over a long tracking run.
        tracked = np.stack([
            (cv2.remap(mask, map_x, map_y, cv2.INTER_LINEAR) > 0.5).astype(np.float32)
            for mask in self.masks
        ])

        # Tracking-failure check. A mask that has vanished or ballooned means the
        # flow has lost the object, and it is worth paying for a fresh segment.
        areas = tracked.reshape(len(tracked), -1).sum(axis=1)
        if np.any(areas < 1):
            return False
        if self.ref_areas is not None:
            ratio = areas / np.maximum(self.ref_areas, 1.0)
            if np.any(ratio > self.mask_area_tolerance) or \
               np.any(ratio < 1.0 / self.mask_area_tolerance):
                return False

        self.masks = tracked
        # Boxes follow the masks instead of staying frozen at the last segmentation.
        self.boxes = self._boxes_from_masks(tracked)
        self.track_count += 1
        return True

    def _detect_scene_change(self, curr_gray: np.ndarray, prev_gray: np.ndarray) -> bool:
        """
        Fraction of pixels whose intensity changed appreciably.

        Comparing mean(|diff|)/255 against 0.3 (the previous approach) requires
        the AVERAGE pixel to move ~76 grey levels, so it effectively never fired.
        """
        if prev_gray is None:
            return True

        diff = cv2.absdiff(curr_gray, prev_gray)
        change_ratio = float(np.mean(diff > self.pixel_change_threshold))

        return change_ratio > self.scene_change_threshold

    def process_frame(self, frame: np.ndarray) -> Dict:
        """Process a single frame with smart segmentation/tracking"""
        start_time = time.time()
        self.frame_count += 1
        h, w = frame.shape[:2]

        # Convert to grayscale for tracking
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Determine if we need to run segmentation
        frames_since_segment = self.frame_count - self.last_segment_frame
        need_segment = (
            self.masks is None or                                # First frame
            frames_since_segment >= self.resegment_interval or   # Periodic refresh
            self._detect_scene_change(gray, self.prev_gray)      # Scene changed
        )

        mode = "SEGMENT"
        if not need_segment:
            if self.use_optical_flow and self.prev_gray is not None:
                t0 = time.time()
                if self._track_masks(gray, self.prev_gray):
                    mode = "TRACK"
                    self.track_times.append(time.time() - t0)
                else:
                    # Tracking lost the object: fall through and re-segment now.
                    self.track_failures += 1
                    mode = "SEGMENT"
            else:
                mode = "SKIP"

        if mode == "SEGMENT":
            t0 = time.time()
            # Resize frame for faster inference (long side -> inference_size)
            scale = self.inference_size / max(h, w)
            frame_small = cv2.resize(frame, None, fx=scale, fy=scale)

            self._run_segmentation(frame_small, frame.shape)
            self.last_segment_frame = self.frame_count
            self.segment_times.append(time.time() - t0)

        self.prev_gray = gray

        # Render masks on frame
        overlay = frame.copy()
        detections = []

        if self.masks is not None and len(self.masks) > 0:
            for i, mask in enumerate(self.masks):
                # Get color and prompt
                if self.class_ids is not None and i < len(self.class_ids):
                    class_id = int(self.class_ids[i])
                else:
                    class_id = 0
                color = self.color_map.get(class_id, [255, 255, 255])
                prompt = self.clothing_prompts[class_id] if class_id < len(self.clothing_prompts) else "object"

                # Masks are already stored at full frame resolution
                mask_binary = (mask > 0.5).astype(np.uint8)

                # Apply colored mask
                colored_mask = np.zeros_like(frame)
                colored_mask[mask_binary > 0] = color
                overlay = cv2.addWeighted(overlay, 1, colored_mask, 0.4, 0)

                # Draw bounding box (already in full-frame coordinates)
                if self.boxes is not None and i < len(self.boxes):
                    x1, y1, x2, y2 = self.boxes[i].astype(int)
                    cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)

                    # Add label
                    label = f"{prompt}"
                    (w_text, h_text), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(overlay, (x1, y1 - h_text - 8), (x1 + w_text, y1), color, -1)
                    cv2.putText(overlay, label, (x1, y1 - 5),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

                detections.append({'prompt': prompt, 'color': color})

        # Calculate FPS (processing only - see the capture loop for end-to-end)
        elapsed = time.time() - start_time
        fps = 1.0 / elapsed if elapsed > 0 else 0
        self.fps_history.append(fps)

        # Add status overlay
        mode_color = (0, 255, 0) if mode == "TRACK" else (0, 165, 255)
        cv2.putText(overlay, f"FPS: {fps:.1f}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(overlay, f"Mode: {mode}", (10, 60),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)
        cv2.putText(overlay, f"Detections: {len(detections)}", (10, 90),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(overlay, f"Segments: {self.segment_count} | Tracks: {self.track_count} | Lost: {self.track_failures}",
                   (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        return {
            'frame': overlay,
            'fps': fps,
            'mode': mode,
            'detections': detections,
            'elapsed': elapsed
        }

print("OptimizedSAM3Segmenter created!")

## 5. Initialize Optimized Segmenter

In [ ]:
# Create optimized segmenter for real-time performance
optimized_segmenter = OptimizedSAM3Segmenter(
    predictor=predictor,
    clothing_prompts=selected_prompts,
    device=device,
    capture_size=(640, 480),      # Webcam capture resolution
    inference_size=320,            # SAM3 inference resolution (lower = faster)
    resegment_interval=150,        # Re-segment every 150 frames (~5 sec)
    scene_change_threshold=0.3,    # Re-segment on 30% scene change
    use_optical_flow=True          # Use optical flow tracking
)

print(f"Optimized segmenter initialized with:")
print(f"Capture: {optimized_segmenter.capture_size}")
print(f"Inference: {optimized_segmenter.inference_size}px") 
print(f"Re-segment every: {optimized_segmenter.resegment_interval} frames")
print(f"Tracking: {'ON' if optimized_segmenter.use_optical_flow else 'OFF'}")

Optimized segmenter initialized with:
Capture: (640, 480)
Inference: 320px
Re-segment every: 15000 frames
Tracking: ON


In [ ]:
def process_webcam_optimized(segmenter, camera_id=0, save_output=False):
    """
    Optimized real-time webcam processing with mask tracking.
    Much faster than running SAM3 on every frame!
    """
    cap = cv2.VideoCapture(camera_id)
    
    if not cap.isOpened():
        print(f"❌ Cannot open camera {camera_id}")
        return
    
    # Set camera properties to match segmenter
    w, h = segmenter.capture_size
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    # Get actual resolution
    actual_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    actual_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"📷 Camera resolution: {actual_w}x{actual_h}")
    
    writer = None
    if save_output:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter('optimized_webcam_output.mp4', fourcc, 30, (actual_w, actual_h))
    
    print(".  Starting optimized webcam...")
    print("   Press 'q' to quit")
    print("   Press 'r' to force re-segmentation")
    print("   Press 's' to toggle save output")
    
    # End-to-end wall clock, including capture, display and encoding
    loop_start = None
    loop_frames = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to grab frame")
                break
            
            if loop_start is None:
                loop_start = time.time()
            
            # Process frame (will auto-segment or track)
            result = segmenter.process_frame(frame)
            result_frame = result['frame']
            
            # Save if requested
            if writer:
                writer.write(result_frame)
            
            # Display
            cv2.imshow('Optimized SAM3 Segmentation (Track + Segment)', result_frame)
            loop_frames += 1
            
            # Handle key presses
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('r'):
                # Force re-segmentation
                segmenter.masks = None
                print("🔄 Forced re-segmentation")
            elif key == ord('s'):
                # Toggle save
                if writer is None:
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    writer = cv2.VideoWriter('optimized_webcam_output.mp4', fourcc, 30, (actual_w, actual_h))
                    print("💾 Recording started")
                else:
                    writer.release()
                    writer = None
                    print("⏹️  Recording stopped")
    
    finally:
        cap.release()
        if writer:
            writer.release()
        cv2.destroyAllWindows()
        
        # Print stats. Segment and track costs are reported separately, since a
        # single blended average just hides how expensive SAM3 actually is.
        elapsed = (time.time() - loop_start) if loop_start else 0
        end_to_end_fps = loop_frames / elapsed if elapsed > 0 else 0
        mean_segment_ms = np.mean(segmenter.segment_times) * 1000 if segmenter.segment_times else 0
        mean_track_ms = np.mean(segmenter.track_times) * 1000 if segmenter.track_times else 0
        
        print(f"\n Session Stats:")
        print(f"   Total frames: {segmenter.frame_count}  ({elapsed:.1f}s wall clock)")
        print(f"   Segmentations: {segmenter.segment_count}  (mean {mean_segment_ms:.0f} ms)")
        print(f"   Tracks: {segmenter.track_count}  (mean {mean_track_ms:.0f} ms)")
        print(f"   Tracking failures: {segmenter.track_failures}")
        print(f"   End-to-end FPS: {end_to_end_fps:.1f}")
        print(f"   Speedup: {segmenter.track_count / max(segmenter.segment_count, 1):.1f}x fewer SAM3 calls")
        if segmenter.frame_count < 100:
            print("   ⚠️  Short run - collect 300+ frames before quoting these numbers")

# Run the webcam segmentation
process_webcam_optimized(optimized_segmenter, camera_id=0, save_output=False)